# EDA — Indicadores Hospitalarios REM20 (Chile 2014–2026)

Exploración inicial de indicadores hospitalarios REM20 cargados en PostgreSQL. El análisis revisa calidad básica, distribuciones, ocupación, letalidad y eficiencia hospitalaria.

In [1]:
import os
from pathlib import Path
import warnings
import urllib.parse

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import dotenv
import sqlalchemy

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    sns.set_theme(style="whitegrid")

def find_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("No se encontró .env en Path.cwd() ni en sus padres.")

ROOT = find_root(Path.cwd())
GRAF = ROOT / "data" / "processed" / "graficos"
GRAF.mkdir(parents=True, exist_ok=True)

dotenv.load_dotenv(ROOT / ".env")
required = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise ValueError("Faltan variables en .env: " + ", ".join(missing))

db_user = urllib.parse.quote_plus(os.getenv("DB_USER"))
db_password = urllib.parse.quote_plus(os.getenv("DB_PASSWORD"))
db_url = (
    f"postgresql+psycopg2://{db_user}:{db_password}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = sqlalchemy.create_engine(db_url, pool_pre_ping=True)

FUENTE = "Fuente: MINSAL REM20 2014-2026"

def guardar_grafico(fig, nombre: str) -> None:
    fig.text(0.99, 0.01, FUENTE, ha="right", va="bottom", fontsize=8, style="italic", color="gray")
    fig.tight_layout()
    fig.savefig(GRAF / f"{nombre}.png", dpi=120, bbox_inches="tight")

In [2]:
df = pd.read_sql("SELECT * FROM rem20.indicadores", engine)
df.shape

(165232, 20)

## Sección 1 — Carga y validación

In [3]:
display(df.shape)
display(df.dtypes)
display(df.describe(include="all"))
display(df.isna().sum().rename("nulos"))

(165232, 20)

periodo                        int64
tipo_pertenencia               int64
cod_sss                        int64
glosa_sss                        str
codigo_establecimiento         int64
establecimiento                  str
cod_area_funcional             int64
area_funcional                   str
mes                            int64
dias_camas_ocupadas            int64
dias_camas_disponibles         int64
dias_estada                    int64
numero_egresos                 int64
egresos_fallecidos             int64
traslados                      int64
indice_ocupacional           float64
promedio_camas_disponible    float64
promedio_dias_estada         float64
letalidad                    float64
indice_rotacion              float64
dtype: object

,periodo,tipo_pertenencia,cod_sss,glosa_sss,codigo_establecimiento,establecimiento,cod_area_funcional,area_funcional,mes,dias_camas_ocupadas,dias_camas_disponibles,dias_estada,numero_egresos,egresos_fallecidos,traslados,indice_ocupacional,promedio_camas_disponible,promedio_dias_estada,letalidad,indice_rotacion
count,165232.000000,165232.0,165232.000000,165232,165232.000000,165232,165232.000000,165232,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000,165232.000000
unique,NaN,NaN,NaN,32,NaN,313,NaN,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,Araucanía Sur,NaN,Hospital Clínico Regional Dr. Guillermo Grant ...,NaN,Área Obstetricia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,11747,NaN,2315,NaN,18247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2019.756778,1.0,15.072274,NaN,115484.484549,NaN,404.683530,NaN,6.384314,553.543557,705.486934,547.000363,75.175190,2.437621,19.656580,64.115962,23.182416,18.962988,7.726181,2.867186
std,3.566342,0.0,7.845355,NaN,9315.634793,NaN,19.146428,NaN,3.459957,884.401013,982.448615,995.646720,125.622395,5.608877,54.679632,35.061850,32.259458,130.905071,19.285678,5.209296
min,2014.000000,1.0,1.000000,NaN,101100.000000,NaN,330.000000,NaN,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2017.000000,1.0,9.000000,NaN,109100.000000,NaN,403.000000,NaN,3.000000,81.000000,180.000000,61.000000,7.000000,0.000000,0.000000,43.690000,6.000000,2.980000,0.000000,0.900000
50%,2020.000000,1.0,15.000000,NaN,115106.000000,NaN,407.000000,NaN,6.000000,248.000000,372.000000,226.000000,26.000000,0.000000,3.000000,72.215000,12.000000,5.940000,0.000000,2.250000
75%,2023.000000,1.0,21.000000,NaN,121114.000000,NaN,414.000000,NaN,9.000000,628.000000,820.000000,602.000000,85.000000,2.000000,18.000000,89.290000,27.000000,12.740000,4.650000,4.000000


periodo                      0
tipo_pertenencia             0
cod_sss                      0
glosa_sss                    0
codigo_establecimiento       0
establecimiento              0
cod_area_funcional           0
area_funcional               0
mes                          0
dias_camas_ocupadas          0
dias_camas_disponibles       0
dias_estada                  0
numero_egresos               0
egresos_fallecidos           0
traslados                    0
indice_ocupacional           0
promedio_camas_disponible    0
promedio_dias_estada         0
letalidad                    0
indice_rotacion              0
Name: nulos, dtype: int64

In [4]:
top_periodos = df["periodo"].value_counts().head(5).rename_axis("periodo").reset_index(name="registros")
display(top_periodos)

top_sss = (
    df.groupby("glosa_sss")["codigo_establecimiento"]
    .nunique()
    .sort_values(ascending=False)
    .head(5)
    .rename("establecimientos_unicos")
    .reset_index()
)
display(top_sss)

dist_area = df["area_funcional"].value_counts(dropna=False).rename_axis("area_funcional").reset_index(name="conteo")
dist_area["porcentaje"] = (dist_area["conteo"] / len(df) * 100).round(2)
display(dist_area)

,periodo,registros
0,2021,13690
1,2020,13655
2,2022,13503
3,2023,13439
4,2024,13371


,glosa_sss,establecimientos_unicos
0,Araucanía Sur,16
1,Del Libertador B.O Higgins,16
2,Del Maule,13
3,Viña del Mar Quillota,12
4,Del Reloncaví,10


,area_funcional,conteo,porcentaje
0,Área Obstetricia,18247,11.04
1,Área Médica Adulto Cuidados Básicos,16579,10.03
2,Área Médico-Quirúrgico Cuidados Básicos,15762,9.54
3,Área Médica Pediátrica Cuidados Básicos,13899,8.41
4,Área Cuidados Intermedios Adultos,10210,6.18
5,Área Médico-Quirúrgico Cuidados Medios,9991,6.05
6,Área Pensionado,9168,5.55
7,Área Cuidados Intensivos Adultos,8188,4.96
8,Área Neonatología Cuidados Básicos,7853,4.75
9,Área Médico-Quirúrgico Pediátrica Cuidados Bás...,6572,3.98


## Sección 2 — Índice Ocupacional

In [5]:
hist_data = df.loc[df["indice_ocupacional"].between(0, 120), "indice_ocupacional"]
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(hist_data, bins=40, kde=True, ax=ax)
ax.set_title("Distribución del índice ocupacional (0-120)")
ax.text(0.5, 1.02, "Los valores >120 (camas prestadas entre servicios) son válidos pero se excluyen del histograma para legibilidad",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=9, color="gray", style="italic")
ax.set_xlabel("Índice ocupacional")
ax.set_ylabel("Registros")
guardar_grafico(fig, "02_hist_indice_ocupacional")
plt.close(fig)

In [6]:
top_areas = df["area_funcional"].value_counts().head(8).index
df_top_areas = df[df["area_funcional"].isin(top_areas)].copy()
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_top_areas, x="area_funcional", y="indice_ocupacional", ax=ax)
ax.set_title("Índice ocupacional por las 8 áreas funcionales con más registros")
ax.set_xlabel("Área funcional")
ax.set_ylabel("Índice ocupacional")
ax.tick_params(axis="x", rotation=35)
guardar_grafico(fig, "02_boxplot_areas")
plt.close(fig)

In [7]:
df_time = df.copy()
df_time["fecha"] = pd.to_datetime(dict(year=df_time["periodo"], month=df_time["mes"], day=1))
serie_ocup = df_time.groupby("fecha", as_index=False)["indice_ocupacional"].mean()
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=serie_ocup, x="fecha", y="indice_ocupacional", ax=ax)
ax.set_title("Promedio mensual nacional del índice ocupacional")
ax.set_xlabel("Fecha")
ax.set_ylabel("Índice ocupacional promedio")
guardar_grafico(fig, "02_serie_indice_ocupacional")
plt.close(fig)

In [8]:
estacionalidad = df.groupby("mes", as_index=False)["indice_ocupacional"].mean().sort_values("mes")
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=estacionalidad, x="mes", y="indice_ocupacional", color="#4C78A8", ax=ax)
ax.set_title("Estacionalidad: promedio de índice ocupacional por mes")
ax.set_xlabel("Mes")
ax.set_ylabel("Índice ocupacional promedio")
guardar_grafico(fig, "02_estacionalidad_mes")
plt.close(fig)

## Sección 3 — Letalidad

In [9]:
df_let = df[df["numero_egresos"] > 0].copy()
df_let["tasa_letalidad"] = df_let["egresos_fallecidos"] / df_let["numero_egresos"] * 100
df_let[["periodo", "establecimiento", "area_funcional", "numero_egresos", "egresos_fallecidos", "tasa_letalidad"]].head()

,periodo,establecimiento,area_funcional,numero_egresos,egresos_fallecidos,tasa_letalidad
0,2014,Hospital Dr Juan Noé Crevanni (Arica),Área Cuidados Intensivos Adultos,9,7,77.777778
1,2014,Hospital Dr Juan Noé Crevanni (Arica),Área Cuidados Intensivos Adultos,7,4,57.142857
2,2014,Hospital Dr Juan Noé Crevanni (Arica),Área Cuidados Intensivos Adultos,15,9,60.000000
3,2014,Hospital Dr Juan Noé Crevanni (Arica),Área Cuidados Intensivos Adultos,7,5,71.428571
4,2014,Hospital Dr Juan Noé Crevanni (Arica),Área Cuidados Intensivos Adultos,11,7,63.636364


In [10]:
letalidad_anual = (
    df_let.groupby("periodo", as_index=False)[["egresos_fallecidos", "numero_egresos"]]
    .sum()
)
letalidad_anual["tasa_letalidad"] = 100 * letalidad_anual["egresos_fallecidos"] / letalidad_anual["numero_egresos"]
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=letalidad_anual, x="periodo", y="tasa_letalidad", marker="o", ax=ax)
ax.axvline(2020, color="red", linestyle="--", linewidth=1)
ax.text(2020.1, ax.get_ylim()[1] * 0.95, "COVID-19", color="red", va="top")
ax.set_title("Tasa de letalidad nacional anual ponderada por egresos")
ax.set_xlabel("Periodo")
ax.set_ylabel("Tasa de letalidad (%)")
guardar_grafico(fig, "03_serie_letalidad_anual")
plt.close(fig)

In [11]:
top_areas_egresos = df_let.groupby("area_funcional")["numero_egresos"].sum().sort_values(ascending=False).head(10).index
heat_base = df_let[df_let["area_funcional"].isin(top_areas_egresos)]
heat_group = heat_base.groupby(["area_funcional", "periodo"], as_index=False)[["egresos_fallecidos", "numero_egresos"]].sum()
heat_group["tasa_letalidad"] = 100 * heat_group["egresos_fallecidos"] / heat_group["numero_egresos"]
heat = heat_group.pivot_table(index="area_funcional", columns="periodo", values="tasa_letalidad")
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heat, cmap="YlOrRd", annot=False, ax=ax)
ax.set_title("Letalidad ponderada por área funcional y periodo")
ax.set_xlabel("Periodo")
ax.set_ylabel("Área funcional")
guardar_grafico(fig, "03_heatmap_letalidad")
plt.close(fig)

In [12]:
top_letalidad_2025 = (
    df_let[df_let["periodo"] == 2025]
    .groupby("establecimiento", as_index=False)[["numero_egresos", "egresos_fallecidos"]]
    .sum()
)
top_letalidad_2025 = top_letalidad_2025[top_letalidad_2025["numero_egresos"] >= 50].copy()
top_letalidad_2025["tasa_letalidad"] = 100 * top_letalidad_2025["egresos_fallecidos"] / top_letalidad_2025["numero_egresos"]
top_letalidad_2025 = top_letalidad_2025.sort_values("tasa_letalidad", ascending=False).head(10)
display(top_letalidad_2025)

,establecimiento,numero_egresos,egresos_fallecidos,tasa_letalidad
106,Hospital San José de Maipo,1358,249,18.335788
102,Hospital San José (Casablanca),236,35,14.830508
73,Hospital Futa Sruka Lawenche Kunko Mapu Mo,76,11,14.473684
78,"Hospital Juana Ross de Edwards (Peñablanca, Vi...",2239,277,12.371594
25,Hospital Comunitario Dr. Roberto Muñoz Urrutia...,517,60,11.605416
124,Hospital Santo Tomás (Limache),1461,159,10.882957
143,Hospital de Gorbea,549,59,10.746812
66,Hospital Dr. Mario Sánchez Vergara (La Calera),1612,171,10.607940
37,Hospital Del Salvador de Peumo,867,90,10.380623
176,Hospital de Salamanca,472,47,9.957627


## Sección 4 — Eficiencia hospitalaria

In [13]:
sample = df.sample(min(5000, len(df)), random_state=42).copy()
top_areas_scatter = df["area_funcional"].value_counts().head(8).index
sample["area_plot"] = np.where(sample["area_funcional"].isin(top_areas_scatter), sample["area_funcional"], "Otras")
p99_estada = sample["promedio_dias_estada"].quantile(0.99)
sample_plot = sample[sample["promedio_dias_estada"] <= p99_estada]
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(data=sample_plot, x="indice_ocupacional", y="promedio_dias_estada", hue="area_plot", s=20, alpha=0.65, ax=ax)
ax.set_title("Índice ocupacional vs promedio de días de estada")
ax.set_xlabel("Índice ocupacional")
ax.set_ylabel("Promedio días de estada")
ax.legend(title="Área funcional", bbox_to_anchor=(1.02, 1), loc="upper left")
guardar_grafico(fig, "04_scatter_ocup_estada")
plt.close(fig)

In [14]:
metricas = ["indice_ocupacional", "promedio_dias_estada", "letalidad", "indice_rotacion", "promedio_camas_disponible"]
corr = df[metricas].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(corr, annot=True, cmap="vlag", center=0, fmt=".2f", ax=ax)
ax.set_title("Matriz de correlación de métricas hospitalarias principales")
ax.set_xlabel("Métrica")
ax.set_ylabel("Métrica")
guardar_grafico(fig, "04_correlacion")
plt.close(fig)

In [15]:
top_estada_2025 = (
    df[df["periodo"] == 2025]
    .groupby("establecimiento", as_index=False)["promedio_dias_estada"]
    .mean()
    .sort_values("promedio_dias_estada", ascending=False)
    .head(10)
)
display(top_estada_2025)

,establecimiento,promedio_dias_estada
91,Hospital Psiquiátrico Dr. Philippe Pinel (Puta...,493.102500
92,"Hospital Psiquiátrico El Peral (Santiago, Puen...",387.748333
190,Instituto Psiquiátrico Dr. José Horwitz Barak ...,343.800972
106,Hospital San José de Maipo,194.954583
39,Hospital Del Salvador de Valparaíso,127.040625
129,Hospital de Carahue,76.628958
43,Hospital Dr. Arturo Hillerns Larrañaga (Saavedra),68.585833
154,Hospital de Maullín,55.433750
59,Hospital Dr. José Arraño (Andacollo),51.532708
30,Hospital Comunitario de Salud Familiar Pedro M...,48.524167
